# Notebook 03 — Data Profiling & Validation
## Class 6 Skill: Profile data; identify quality issues; define business-oriented rules; record assumptions

This notebook covers:
- Column-level profiling (null rates, ranges, distributions)
- Identifying meaningful quality issues (not just "is it null")
- Defining 8 business-oriented validation rules
- Recording all assumptions and limitations — NOT silently fixing them

> FDE mindset: Every quality decision is a business decision. Document what you chose and why.

In [ ]:
import sys
sys.path.insert(0, '../pipeline')

import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for script compatibility
import matplotlib.pyplot as plt

from ingest import ingest_trips, ingest_zones, ingest_vendors
from validate import profile_dataframe, validate, save_validation_log

trips_df = ingest_trips()
print(f'Loaded {len(trips_df):,} raw trips.')

## 1. Column-Level Profiling

In [ ]:
profile = profile_dataframe(trips_df)
print('Column profiling results:')
print(profile.to_string(index=False))

## 2. Meaningful Quality Issues Identified

From profiling, these are the quality issues that matter for the business (not every minor anomaly):

| Issue | Column(s) | Scale | Business Impact |
|---|---|---|---|
| Zero or negative fare amounts | fare_amount | ~0.1-0.5% of trips | Would inflate or deflate revenue KPI |
| Trip duration < 1 min or > 5 hrs | pickup/dropoff datetime | ~1-2% | Anomalous trips distort avg duration |
| Avg speed > 80 mph or < 5 mph | derived | ~1-3% | Likely GPS or meter error |
| Trips timestamped in wrong year | tpep_pickup_datetime | small | Data from wrong month in file |
| NULL passenger_count | passenger_count | ~1-5% | Cannot verify trip capacity rules |
| Tip = NULL (cash payments) | tip_amount | ~30% of cash trips | Systematic, not random — excluded from avg |

> **Key distinction:** Tip being NULL for cash payments is *systematic* (by design — cash tips are not captured), not random. We document this as a Known issue, not a data quality bug.

## 3. Business-Oriented Validation Rules

Rules are defined in `validate.py` and driven by thresholds in `config.py`.

| Rule | Code | Rationale |
|---|---|---|
| 1 min ≤ duration ≤ 300 min | VR-01 | Under 1 min = likely test/error; over 5 hrs = likely data error or parking |
| 0.1 mi ≤ distance ≤ 200 mi | VR-02 | TLC operates within NYC metro; 200 mi is outside any plausible airport run |
| 5 mph ≤ avg_speed ≤ 80 mph | VR-03 | Below 5 mph = stationary meter; above 80 mph = GPS error on NYC roads |
| fare_amount > 0 | VR-04 | Zero fare = comp trip (not revenue); negative fare = correction entry |
| 1 ≤ passenger_count ≤ 6 | VR-05 | TLC max capacity; 0 passengers = meter test or data error |
| VendorID in {1, 2} | VR-06 | Only two licensed Yellow Taxi data providers per TLC |
| pickup < dropoff, both in 2024 | VR-07 | Inverted timestamps = data entry error; wrong year = wrong month in file |
| LocationID > 0, not null | VR-08 | Cannot attribute trip to a zone without a valid ID |

**Important:** Zero-fare and negative-fare trips are NOT silently dropped. They are classified into `ZERO_FARE` and `NEGATIVE_FARE` status codes so the business can count them separately if needed.

In [ ]:
print('Applying validation rules ...')
valid_df, reject_df = validate(trips_df)

print(f'\nValidation results:')
print(f'  Valid trips   : {len(valid_df):>10,} ({100*len(valid_df)/len(trips_df):.1f}%)')
print(f'  Rejected trips: {len(reject_df):>10,} ({100*len(reject_df)/len(trips_df):.1f}%)')

In [ ]:
# Rejection breakdown
reason_counts = (
    reject_df['rejection_reason']
    .str.split(' | ')
    .explode()
    .value_counts()
    .reset_index()
)
reason_counts.columns = ['rejection_reason', 'count']
reason_counts['pct_of_all_trips'] = (100 * reason_counts['count'] / len(trips_df)).round(2)
print('Rejection breakdown by rule:')
print(reason_counts.to_string(index=False))

In [ ]:
# Save rejection log
save_validation_log(reject_df)
print('Rejection log saved to outputs/validation_log.csv')

## 4. Assumptions, Known Issues, and Limitations

*(These are recorded here, not silently fixed in code)*

| Category | Item |
|---|---|
| **Known** | Cash payment tips are systematically NULL — this is by design, not a bug |
| **Known** | Some trips have VendorID = NULL — treated as unknown vendor (VR-06) |
| **Known** | LocationID 264 and 265 are labelled 'Unknown' in the zone lookup |
| **Unknown** | Whether trips with zero fare are comp rides or meter tests — we flag them rather than deciding |
| **Assumption** | Thresholds (1 min, 300 min, 80 mph) were chosen based on TLC data dictionary guidance and domain knowledge |
| **Assumption** | A trip timestamped outside January 2024 in the Jan-2024 file is a data error, not intentional |
| **Limitation** | Speed is calculated as distance / duration, not actual road distance — underestimates highway speeds |
| **Limitation** | Rejection thresholds in config.py are static — a future version could use percentile-based dynamic thresholds |

## 5. Distribution Checks on Validated Data

In [ ]:
print('Distribution of validated trip durations (minutes):')
print(valid_df['duration_minutes'].describe().round(2))

print('\nDistribution of validated fare amounts ($):')
print(valid_df['fare_amount'].describe().round(2))

print('\nDistribution of validated avg speed (mph):')
print(valid_df['avg_speed_mph'].describe().round(2))